# Classification: Temporal Features

Instead of averaging metrics across all trials, each session is split into the first third (early) and the last third (late) of trials. For each segment we compute the per-session mean of each metric, then add delta features (late − early) to capture within-session change - attention deterioration, fatigue, habituation. After computing per-session early / late values, features are averaged per user and deltas are recomputed at the user level to reduce within-subject measurement noise.

Three tasks are tested:
- binary classification (depressed vs not)
- multi-class (severity groups)
- regression (predict score directly)

Every task is run on two depression scales:
- **PHQ-9** (Patient Health Questionnaire, 9 items, range 0–27)
- **BDI-II** (Beck Depression Inventory, 21 items, range 0–63)

Running both scales provides a convergent-validity check: if the same gaze features predict both questionnaires similarly, the signal reflects the depression construct rather than questionnaire-specific content.

Three models are compared:
- Logistic / Ridge Regression
- Random Forest
- XGBoost

Across three feature sets:
- all temporal (early + late + delta)
- theory-driven temporal (early + late + delta)
- delta only

In [0]:
%pip install xgboost -q

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np

from src.evaluation.classification import (
    prepare_data, run_classification_binary, run_classification_multiclass, run_regression,
    plot_best_classification_binary, plot_best_classification_multiclass, plot_best_regression,
    plot_summary, plot_feature_importance, build_comparison_table,
)

## 1. Build session-level dataset

### 1.1 Build temporal session features

In [0]:
from src.features.session_aggregation import (
    build_temporal_aggregation,
    add_delta_columns,
)

VALID_PAIRS = ["negative_vs_positive", "negative_vs_neutral", "neutral_vs_positive"]
DURATION_MIN_MS = 500
DURATION_MAX_MS = 10000

scene_metrics = spark.table("anima.scene_metrics")
df_forms = spark.table("anima.forms")

df_stimulus = (scene_metrics
    .filter(F.col("scene_type") == "stimulus")
    .filter(F.col("scene_quality_valid") == True)
    .filter(F.col("scene_valence_pair").isin(VALID_PAIRS))
    .filter(F.col("duration_ms").between(DURATION_MIN_MS, DURATION_MAX_MS)))

print(f"Clean stimulus scenes: {df_stimulus.count()}")

w = Window.partitionBy("session_id").orderBy("scene_index")
numbered = (df_stimulus
    .withColumn("trial_num", F.row_number().over(w))
    .withColumn("n_trials", F.count("*").over(Window.partitionBy("session_id"))))

split = (numbered
    .filter((F.col("trial_num") <= F.col("n_trials") / 3) |
            (F.col("trial_num") >  F.col("n_trials") * 2 / 3))
    .withColumn("is_early", F.col("trial_num") <= F.col("n_trials") / 3))

agg = build_temporal_aggregation()
print(f"Spark aggregation expressions: {len(agg.exprs)}")

session_metrics = split.groupBy("session_id").agg(*agg.exprs)

df_joined = session_metrics.join(
    df_forms.select("session_id", "uid", "phq9_score", "bdi_score"),
    on="session_id", how="inner",
)

df = df_joined.toPandas()

df = add_delta_columns(df, agg)

print(f"Sessions: {len(df)}, Users: {df['uid'].nunique()}")
print(f"Total features (incl deltas): {len(agg.all_columns)}")

### 1.2 Aggregate to user level

In [0]:
FEATURE_COLS = agg.all_columns
NUMERIC_COLS = FEATURE_COLS + ["phq9_score", "bdi_score"]
df = df.groupby("uid", as_index=False)[NUMERIC_COLS].mean()

print(f"After per-user aggregation: {len(df)} users")
print(f"PHQ-9: min={df['phq9_score'].min():.1f}, max={df['phq9_score'].max():.1f}, mean={df['phq9_score'].mean():.1f}")
print(f"BDI-II: min={df['bdi_score'].min():.1f}, max={df['bdi_score'].max():.1f}, mean={df['bdi_score'].mean():.1f}")

## 2. Define targets and feature sets

### 2.1 PHQ-9 targets

In [0]:
df["phq9_depressed"] = (df["phq9_score"] >= 10).astype(int)

def phq9_severity_from_score(s):
    if s <= 4:  return 0
    if s <= 9:  return 1
    if s <= 14: return 2
    if s <= 19: return 3
    return 4

df["phq9_severity_class"] = df["phq9_score"].apply(phq9_severity_from_score)

print("PHQ-9")
print(f"Binary (>=10): {df['phq9_depressed'].value_counts().sort_index().to_dict()}")
print(f"Multi-class: {df['phq9_severity_class'].value_counts().sort_index().to_dict()}")

### 2.2 BDI-II targets

In [0]:
df["bdi_depressed"] = (df["bdi_score"] >= 14).astype(int)

def bdi_severity_from_score(s):
    if s <= 13: return 0
    if s <= 19: return 1
    if s <= 28: return 2
    return 3

df["bdi_severity_class"] = df["bdi_score"].apply(bdi_severity_from_score)

print("BDI-II")
print(f"Binary (>=14): {df['bdi_depressed'].value_counts().sort_index().to_dict()}")
print(f"Multi-class: {df['bdi_severity_class'].value_counts().sort_index().to_dict()}")

### 2.3 Feature sets

In [0]:
ALL_FEATURES = agg.all_columns

THEORY_PAIR_SPLIT_METRICS = [
    "dwell_neg__neg_vs_pos",
    "fix_prop_neg__neg_vs_pos",
    "dwell_pos__pos_vs_neu",
    "fix_prop_pos__pos_vs_neu",
    "revisit_neg__neg_vs_pos",
    "revisit_neg__neg_vs_neu",
]
THEORY_BIAS_METRICS = [
    "bias_neg_vs_pos",
    "bias_neg_vs_neu",
    "bias_pos_vs_neu",
]
THEORY_SIMPLE_METRICS = [
    "scanpath_length",
    "gaze_transition_entropy",
]
THEORY_FIRST_FIX_METRICS= [
    "first_fix_prob_neg__neg_vs_pos",
    "first_fix_prob_pos__pos_vs_neu",
]

THEORY_FEATURES = []
for metric in (THEORY_SIMPLE_METRICS + THEORY_BIAS_METRICS + THEORY_PAIR_SPLIT_METRICS + THEORY_FIRST_FIX_METRICS):
    THEORY_FEATURES.extend([f"{metric}_early", f"{metric}_late", f"{metric}_delta"])

DELTA_ONLY_FEATURES = [c for c in ALL_FEATURES if c.endswith("_delta")]

missing = [f for f in THEORY_FEATURES if f not in ALL_FEATURES]
assert not missing, f"Theory features not in ALL_FEATURES: {missing}"

FEATURE_SETS = {
    "All temporal": ALL_FEATURES,
    "Theory-driven temporal": THEORY_FEATURES,
    "Delta only": DELTA_ONLY_FEATURES,
}

for name, feats in FEATURE_SETS.items():
    print(f"{name}: {len(feats)} features")

## 3. Prepare data

In [0]:
target_cols = [
    "phq9_score", "phq9_depressed", "phq9_severity_class",
    "bdi_score", "bdi_depressed", "bdi_severity_class",
]
df_clean, groups = prepare_data(df, FEATURE_SETS, target_cols)

## 4. Binary classification

### 4.1 PHQ-9 cutoff (≥ 10)

#### 4.1.1 Run classification

In [0]:
y_phq9_binary = df_clean["phq9_depressed"].values
phq9_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups)

#### 4.1.2 Results

In [0]:
print(phq9_binary_df.to_string(index=False))

#### 4.1.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_phq9_binary, groups, phq9_binary_df)

### 4.2 PHQ-9 extremes (minimal vs severe)

#### 4.2.1 Run classification

In [0]:
df_phq9_extreme = df_clean[(df_clean["phq9_score"] <= 4) | (df_clean["phq9_score"] >= 20)].copy()
df_phq9_extreme["phq9_extreme"] = (df_phq9_extreme["phq9_score"] >= 20).astype(int)

groups_phq9_extreme = df_phq9_extreme["uid"].values

print(f"PHQ-9 extremes dataset: {len(df_phq9_extreme)} users")
print(f"Minimal (<=4): {(df_phq9_extreme['phq9_extreme'] == 0).sum()}")
print(f"Severe (>=20): {(df_phq9_extreme['phq9_extreme'] == 1).sum()}")
print()

y_phq9_extreme = df_phq9_extreme["phq9_extreme"].values
phq9_extreme_df = run_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme)

#### 4.2.2 Results

In [0]:
print(phq9_extreme_df.to_string(index=False))

#### 4.2.3 Best model

In [0]:
plot_best_classification_binary(df_phq9_extreme, FEATURE_SETS, y_phq9_extreme, groups_phq9_extreme, phq9_extreme_df)

### 4.3 BDI-II cutoff (≥ 14)

#### 4.3.1 Run classification

In [0]:
y_bdi_binary = df_clean["bdi_depressed"].values
bdi_binary_df = run_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups)

#### 4.3.2 Results

In [0]:
print(bdi_binary_df.to_string(index=False))

#### 4.3.3 Best model

In [0]:
plot_best_classification_binary(df_clean, FEATURE_SETS, y_bdi_binary, groups, bdi_binary_df)

### 4.4 BDI-II extremes (minimal vs severe)

#### 4.4.1 Run classification

In [0]:
df_bdi_extreme = df_clean[(df_clean["bdi_score"] <= 13) | (df_clean["bdi_score"] >= 29)].copy()
df_bdi_extreme["bdi_extreme"] = (df_bdi_extreme["bdi_score"] >= 29).astype(int)

groups_bdi_extreme = df_bdi_extreme["uid"].values

print(f"BDI-II extremes dataset: {len(df_bdi_extreme)} users")
print(f"Minimal (<=13): {(df_bdi_extreme['bdi_extreme'] == 0).sum()}")
print(f"Severe (>=29): {(df_bdi_extreme['bdi_extreme'] == 1).sum()}")
print()

y_bdi_extreme = df_bdi_extreme["bdi_extreme"].values
bdi_extreme_df = run_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme)

#### 4.4.2 Results

In [0]:
print(bdi_extreme_df.to_string(index=False))

#### 4.4.3 Best model

In [0]:
plot_best_classification_binary(df_bdi_extreme, FEATURE_SETS, y_bdi_extreme, groups_bdi_extreme, bdi_extreme_df)

## 5. Multi-class classification

### 5.1 PHQ-9 (5 severity classes)

#### 5.1.1 Run classification

In [0]:
PHQ9_LABELS = ["Minimal", "Mild", "Moderate", "Moderately Severe", "Severe"]
y_phq9_multi = df_clean["phq9_severity_class"].values.astype(int)
phq9_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups)

#### 5.1.2 Results

In [0]:
print(phq9_multi_df.to_string(index=False))

#### 5.1.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_phq9_multi, groups, phq9_multi_df, PHQ9_LABELS)

### 5.2 BDI-II (4 severity classes)

#### 5.2.1 Run classification

In [0]:
BDI_LABELS = ["Minimal", "Mild", "Moderate", "Severe"]
y_bdi_multi = df_clean["bdi_severity_class"].values.astype(int)
bdi_multi_df = run_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups)

#### 5.2.2 Results

In [0]:
print(bdi_multi_df.to_string(index=False))

#### 5.2.3 Best model

In [0]:
plot_best_classification_multiclass(df_clean, FEATURE_SETS, y_bdi_multi, groups, bdi_multi_df, BDI_LABELS)

## 6. Regression

### 6.1 PHQ-9 score prediction

#### 6.1.1 Run regression

In [0]:
y_phq9_reg = df_clean["phq9_score"].values
phq9_reg_df = run_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups)

#### 6.1.2 Results

In [0]:
print(phq9_reg_df.to_string(index=False))

#### 6.1.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_phq9_reg, groups, phq9_reg_df, score_name="PHQ-9")

### 6.2 BDI-II score prediction

#### 6.2.1 Run regression

In [0]:
y_bdi_reg = df_clean["bdi_score"].values
bdi_reg_df = run_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups)

#### 6.2.2 Results

In [0]:
print(bdi_reg_df.to_string(index=False))

#### 6.2.3 Best model

In [0]:
plot_best_regression(df_clean, FEATURE_SETS, y_bdi_reg, groups, bdi_reg_df, score_name="BDI-II")

## 7. PHQ-9 vs BDI-II convergent-validity comparison

In [0]:
phq9_results = {
    "Binary cutoff": (phq9_binary_df, "auc_roc"),
    "Binary extremes": (phq9_extreme_df, "auc_roc"),
    "Multi-class": (phq9_multi_df, "f1_weighted"),
    "Regression": (phq9_reg_df, "r2"),
}
bdi_results = {
    "Binary cutoff": (bdi_binary_df, "auc_roc"),
    "Binary extremes": (bdi_extreme_df, "auc_roc"),
    "Multi-class": (bdi_multi_df, "f1_weighted"),
    "Regression": (bdi_reg_df, "r2"),
}

comparison_df = build_comparison_table(phq9_results, bdi_results)
print(comparison_df.to_string(index=False))

## 8. Feature importance

In [0]:
plot_feature_importance(df_clean, ALL_FEATURES, y_phq9_binary, title="Feature importance (PHQ-9 binary, all temporal)")

## 9. Summary

### 9.1 PHQ-9

In [0]:
feature_order = list(FEATURE_SETS.keys())
plot_summary(phq9_binary_df, phq9_multi_df, phq9_reg_df, feature_order, title="PHQ-9, temporal features")

### 9.2 BDI-II

In [0]:
plot_summary(bdi_binary_df, bdi_multi_df, bdi_reg_df, feature_order, title="BDI-II, temporal features")